<a href="https://colab.research.google.com/github/BhavyaMShah/Portfolio-Analysis/blob/main/Portfolio_Valuation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import pandas as pd

def DCF_upload(data):
  intrinsic_values_list = []

  for d in range(0,len(data)):
    Tax = float(data['Tax'][d]) / 100
    EBIT = float(data['EBIT'][d])
    Dep = float(data['Dep'][d])
    Amo = float(data['Amo'][d])
    CAPEX = float(data['CAPEX'][d])
    NWC = float(data['NWC'][d])

    Risk_free = float(data['Risk_free'][d]) / 100
    Rm = float(data['Ex_Market_Ret1'][d]) / 100
    Beta = float(data['Beta'][d])
    CoD = float(data['CoD1'][d]) / 100

    Debt = float(data['Debt'][d])
    Equity = float(data['Equity'][d])

    Growth = float(data['Growth1'][d]) / 100
    Years = int(data['Years'][d])
    Terminal_Growth = float(data['Terminal_Growth1'][d]) / 100
    Shares = float(data['Shares'][d])

    if EBIT <= 0 or Shares <= 0 or (Debt + Equity) <= 0:
        continue
    #Free Cash Flow
    if Tax <= 0 or EBIT <= 0:
        print("Error: Tax rate or EBIT is zero or negative. Skipping this company.")
        continue
    else:
        FCFF_0 = EBIT * (1 - Tax) + Dep + Amo - CAPEX - NWC

    #CAPM
    CoE = Risk_free + Beta * (Rm - Risk_free)

    # WACC
    Total_Cap = Debt + Equity
    w_e = Equity / Total_Cap
    w_d = Debt / Total_Cap

    WACC = w_e * CoE + w_d * CoD * (1 - Tax)

    if WACC <= Terminal_Growth:
      continue

    # FCFF FORECAST
    FCFF_list = []
    FCFF_prev = FCFF_0

    for year in range(1, Years + 1):
        FCFF_prev *= (1 + Growth)
        FCFF_list.append(FCFF_prev)

    # DISCOUNT FCFF
    PV_FCFF = [
        fcff / (1 + WACC) ** t
        for t, fcff in enumerate(FCFF_list, start=1)]

    # TERMINAL VALUE
    Terminal_Value = FCFF_list[-1] * (1 + Terminal_Growth) / (WACC - Terminal_Growth)
    PV_Terminal_Value = Terminal_Value / (1 + WACC) ** Years

    # ENTERPRISE & EQUITY VALUE
    Enterprise_Value = sum(PV_FCFF) + PV_Terminal_Value
    Equity_Value = Enterprise_Value - Debt
    Intrinsic_Value_per_Share = Equity_Value / Shares

    # SHARE VALUE
    Equity_Value = Enterprise_Value - Debt
    Intrinsic_Value_per_Share = Equity_Value / Shares
    intrinsic_values_list.append(Intrinsic_Value_per_Share)


    # STORING IN DATAFRAME
    data.at[d, 'Enterprise_Value'] = Enterprise_Value
    data.at[d, 'Intrinsic_Value_per_Share'] = Intrinsic_Value_per_Share
    data.at[d, 'Terminal_Value'] = Terminal_Value
    data.at[d, 'PV_Terminal_Value'] = PV_Terminal_Value

  print("\nFINAL DCF SUMMARY\n")
  if 'Company' in data.columns:
      print(data[['Company','Enterprise_Value','Intrinsic_Value_per_Share']].set_index('Company'))
  else:
      print(data[['Enterprise_Value', 'Intrinsic_Value_per_Share']])
      print("Note: 'Company' column not found in the uploaded file. Displaying only calculated values.")

  return data

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)
pd.set_option('display.colheader_justify', 'left')
pd.set_option('display.float_format', '{:,.2f}'.format)

In [24]:
from google.colab import files
import pandas as pd
import os

def upload_data():

  files.upload()

  print(os.listdir())
  print('\n')
  file_name = str(input("Your File Name: "))

  # Read file
  df = None
  if file_name.endswith('.csv'):
      df = pd.read_csv(file_name)
  elif file_name.endswith(('.xls', '.xlsx')):
      df = pd.read_excel(file_name)
  else:

      print(f"Error: Unsupported file type for '{file_name}'. Please provide a .csv, .xls, or .xlsx file.")
      return

  global dcf_results_df
  dcf_results_df = DCF_upload(df)

In [30]:
def collect_owner_shares(df_results):

    # Ensure columns exist
    df_results['User_Owned_Shares'] = df_results.get('User_Owned_Shares', 0).fillna(0).astype(float)
    df_results['Market_price'] = df_results.get('Market_price', 0).fillna(0).astype(float)

    # Vectorized calculations
    df_results['Intrinsic_Portfolio'] = (
        df_results['User_Owned_Shares'] * df_results['Intrinsic_Value_per_Share']
    )
    df_results['Real_Portfolio'] = (
        df_results['User_Owned_Shares'] * df_results['Market_price']
    )

    # Totals
    total_intrinsic = df_results['Intrinsic_Portfolio'].sum()
    total_real = df_results['Real_Portfolio'].sum()

    # Display formatting FIRST
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    pd.set_option('display.float_format', '{:,.2f}'.format)

    print("\n--- Updated DCF Summary with Your Holdings ---\n")

    if 'Company' in df_results.columns:
        print(
            df_results[
                ['Company','Intrinsic_Value_per_Share','Market_price','User_Owned_Shares',
                 'Intrinsic_Portfolio','Real_Portfolio']].set_index('Company'))

    print(f"\nTotal Intrinsic Value: ${total_intrinsic:,.2f}")
    print(f"Total Real Portfolio Value: ${total_real:,.2f}")

def Portfolio_Valuation():
  upload_data()
  collect_owner_shares(dcf_results_df)